# Week6

## 문제2) 서울시 지하철역 정보 수집

In [4]:
import os
import src.mylib

keyPath=os.path.join(os.getcwd(),'src','key.properties')
key=src.mylib.getKey(keyPath)
KEY=str(key['dataseoul'])

In [5]:
# api
TYPE='json'
SERVICE='SearchSTNBySubwayLineInfo'
START_INDEX=str(1)
END_INDEX=str(10)
LINE_NUM=str(2)

##### URL만들기

In [6]:
#문제에서 이 url에서 데이터를 받으시오하면 할 줄 알아야 함
## 마지막 // 2개 의마
params="/".join([KEY,TYPE,SERVICE,START_INDEX,END_INDEX,'','',LINE_NUM])
print(params[30:])

/json/SearchSTNBySubwayLineInfo/1/10///2


In [7]:
#오류 버전
#---------------------------------------------------------------------
import urllib
_url='http://openAPI.seoul.go.kr:8088/'
url=urllib.parse.urljoin(_url,params)
print(url) # backslashes removed
#인자(전철역코드,전철역명)가 생략됐기 때문에 검색 오류 발생

http://openAPI.seoul.go.kr:8088/444178666c6a656835366d43634844/json/SearchSTNBySubwayLineInfo/1/10/2


In [10]:
#정상 버전
#---------------------------------------------------------------------
_url='http://openAPI.seoul.go.kr:8088'
url="/".join([_url,params])

print (url)

http://openAPI.seoul.go.kr:8088/444178666c6a656835366d43634844/json/SearchSTNBySubwayLineInfo/1/10///2


##### URL 호출

In [11]:
import requests # http로 요청 및 받음

r=requests.get(url)
stations=r.json()

In [12]:
print(stations) #json형식으로 받았기 때문에 Dict형태(K:V)

{'SearchSTNBySubwayLineInfo': {'list_total_count': 799, 'RESULT': {'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다'}, 'row': [{'STATION_CD': '1707', 'STATION_NM': '명학', 'STATION_NM_ENG': 'Myeonghak', 'LINE_NUM': '01호선', 'FR_CODE': 'P148', 'STATION_NM_CHN': '鸣鹤', 'STATION_NM_JPN': 'ミョンハク'}, {'STATION_CD': '1709', 'STATION_NM': '군포', 'STATION_NM_ENG': 'Gunpo', 'LINE_NUM': '01호선', 'FR_CODE': 'P150', 'STATION_NM_CHN': '军浦', 'STATION_NM_JPN': 'クンポ'}, {'STATION_CD': '1815', 'STATION_NM': '부개', 'STATION_NM_ENG': 'Bugae', 'LINE_NUM': '01호선', 'FR_CODE': '151', 'STATION_NM_CHN': '富开', 'STATION_NM_JPN': 'プゲ'}, {'STATION_CD': '1814', 'STATION_NM': '소사', 'STATION_NM_ENG': 'Sosa', 'LINE_NUM': '01호선', 'FR_CODE': '147', 'STATION_NM_CHN': '素砂', 'STATION_NM_JPN': 'ソサ'}, {'STATION_CD': '1813', 'STATION_NM': '구일', 'STATION_NM_ENG': 'Guil', 'LINE_NUM': '01호선', 'FR_CODE': '142', 'STATION_NM_CHN': '九一', 'STATION_NM_JPN': 'クイル'}, {'STATION_CD': '1812', 'STATION_NM': '인천', 'STATION_NM_ENG': 'Incheon', 'LINE_NUM': '

In [15]:
stations['SearchSTNBySubwayLineInfo']['row'][1:4] #우리가 갖고싶은 data는 SearchSTNBySubwayLineInfo의 row임.

[{'STATION_CD': '1709',
  'STATION_NM': '군포',
  'STATION_NM_ENG': 'Gunpo',
  'LINE_NUM': '01호선',
  'FR_CODE': 'P150',
  'STATION_NM_CHN': '军浦',
  'STATION_NM_JPN': 'クンポ'},
 {'STATION_CD': '1815',
  'STATION_NM': '부개',
  'STATION_NM_ENG': 'Bugae',
  'LINE_NUM': '01호선',
  'FR_CODE': '151',
  'STATION_NM_CHN': '富开',
  'STATION_NM_JPN': 'プゲ'},
 {'STATION_CD': '1814',
  'STATION_NM': '소사',
  'STATION_NM_ENG': 'Sosa',
  'LINE_NUM': '01호선',
  'FR_CODE': '147',
  'STATION_NM_CHN': '素砂',
  'STATION_NM_JPN': 'ソサ'}]

In [16]:
for e in stations['SearchSTNBySubwayLineInfo']['row']:
    print ("{0:4s}\t{1:15s}\t{2:3s}\t{3:2s}".format(e['STATION_CD'], e['STATION_NM'], e['FR_CODE'], e['LINE_NUM']))

1707	명학             	P148	01호선
1709	군포             	P150	01호선
1815	부개             	151	01호선
1814	소사             	147	01호선
1813	구일             	142	01호선
1812	인천             	161	01호선
1811	동인천            	160	01호선
1810	제물포            	158	01호선
1809	주안             	156	01호선
1808	동암             	154	01호선


##### 일괄 실행

In [20]:
%%writefile src/ds_open_subway_json.py
import os
import requests
import urllib
import mylib #src.mylib이 아닌 그냥 mylib #%%writefile src를 했기 때문에 이미 src 하위 폴더에 생김

def doIt():
    keyPath=os.path.join(os.getcwd(), 'src', 'key.properties')
    key=mylib.getKey(keyPath)
    # (1) make params with resource IDs
    KEY=str(key['dataseoul'])
    TYPE='json'
    #OLD: SERVICE='SearchSTNBySubwayLineService'
    SERVICE='SearchSTNBySubwayLineInfo'
    LINE_NUM=str(2)
    START_INDEX=str(1)
    END_INDEX=str(10)
    #params=os.path.join(KEY,TYPE,SERVICE,START_INDEX,END_INDEX,LINE_NUM)
    params="/".join([KEY,TYPE,SERVICE,START_INDEX,END_INDEX,'','',LINE_NUM])
    # (2) make a full url
    _url='http://openAPI.seoul.go.kr:8088' #NOTE slash: do not use 'http://openAPI.seoul.go.kr:8088/'
    #url=urllib.parse.urljoin(_url,params)
    url="/".join([_url,params])
    #print(url)
    # (3) get data
    r=requests.get(url)
    #print(r)
    stations=r.json()
    #print(stations)
    for e in stations['SearchSTNBySubwayLineInfo']['row']:
        print (u"{0:4s}\t{1:15s}\t{2:3s}\t{3:2s}".format(e['STATION_CD'], e['STATION_NM'], e['FR_CODE'], e['LINE_NUM']))

if __name__ == "__main__":  ##관영어구로 함수 호출(메인 함수 역할)
    doIt()

Writing src/ds_open_subway_json.py


In [22]:
!python src/ds_open_subway_json.py #202609

1707	명학             	P148	01호선
1709	군포             	P150	01호선
1815	부개             	151	01호선
1814	소사             	147	01호선
1813	구일             	142	01호선
1812	인천             	161	01호선
1811	동인천            	160	01호선
1810	제물포            	158	01호선
1809	주안             	156	01호선
1808	동암             	154	01호선


In [23]:
stations['SearchSTNBySubwayLineInfo']['list_total_count']

799

In [24]:
startIndex=10
if(stations["SearchSTNBySubwayLineInfo"]["RESULT"]["CODE"]=="INFO-000"):
    print("Error: "+str(startIndex))

Error: 10


In [25]:
%%writefile src/ds_open_subway_iter_json.py
#!/usr/bin/env python
# coding: utf-8
import os
import requests
import urllib
import mylib # NO! src.mylib

def doIt():
    keyPath=os.path.join(os.getcwd(), 'src', 'key.properties')
    key=mylib.getKey(keyPath)
    # (1) make params with resource IDs
    KEY=str(key['dataseoul'])
    TYPE='json'
    #OLD: SERVICE='SearchSTNBySubwayLineService'
    SERVICE='SearchSTNBySubwayLineInfo'
    LINE_NUM=str(2)
    START_INDEX=str(1)
    END_INDEX=str(10)
    startIndex=1
    endIndex=10
    list_total_count=0    # set later
    while True:
        START_INDEX=str(startIndex)
        END_INDEX=str(endIndex)
        #params=os.path.join(KEY,TYPE,SERVICE,START_INDEX,END_INDEX,LINE_NUM)
        params="/".join([KEY,TYPE,SERVICE,START_INDEX,END_INDEX,'','',LINE_NUM])
        # (2) make a full url
        _url='http://openAPI.seoul.go.kr:8088' #NOTE slash: do not use 'http://openAPI.seoul.go.kr:8088/'
        #url=urllib.parse.urljoin(_url,params)
        url="/".join([_url,params])
        #print(url)
        # (3) get data
        r=requests.get(url)
        #print(r)
        stations=r.json()
        #print(stations)
        if(startIndex==1):
            list_total_count=stations['SearchSTNBySubwayLineInfo']['list_total_count']
            print("- Total Count: ", list_total_count)
        if(stations["SearchSTNBySubwayLineInfo"]["RESULT"]["CODE"]=="INFO-000"):
            for e in stations['SearchSTNBySubwayLineInfo']['row'][::10]:
                print (u"{0:3d}\t{1:4s}\t{2:15s}\t{3:3s}\t{4:2s}".format(
                    startIndex, e['STATION_CD'], e['STATION_NM'], e['FR_CODE'], e['LINE_NUM']
                ))
            startIndex+=10
            endIndex+=10
        else:
            print("Error: "+str(startIndex))
        if(endIndex > list_total_count):
            print("----- Ending endIndex=",endIndex)
            break  # exit from the while loop

if __name__ == "__main__":
    doIt()

Writing src/ds_open_subway_iter_json.py


In [27]:
!python src/ds_open_subway_iter_json.py #202609

- Total Count:  799
  1	1707	명학             	P148	01호선
 11	1807	백운             	153	01호선
 21	1712	화서             	P154	01호선
 31	1403	아산             	P172	01호선
 41	0156	신설동            	126	01호선
 51	1803	역곡             	146	01호선
 61	1702	가산디지털단지        	P142	01호선
 71	1913	동두천중앙          	103	01호선
 81	1717	세마             	P158	01호선
 91	1909	양주             	107	01호선
101	1817	도원             	159	01호선
111	0247	도림천            	234-1	02호선
121	0241	이대             	241	02호선
131	0237	당산             	237	02호선
141	0227	낙성대            	227	02호선
151	0203	을지로3가          	203	02호선
161	0326	압구정            	336	03호선
171	0339	수서             	349	03호선
181	0316	독립문            	326	03호선
191	1955	마두             	312	03호선
201	0433	사당             	433	04호선
211	0405	진접             	405	04호선
221	1754	상록수            	448	04호선
231	0415	미아             	415	04호선
241	0419	한성대입구          	419	04호선
251	2555	둔촌동            	P549	05호선
261	2551	굽은다리           	550	05호선
271	2561	마천             	P555	05호선
281	2538	청구        

#### XML파싱

In [30]:
#json : 인자를 변경할 떄 편리, xml은 한 줄에 url을 넣기 때문에 간편
import lxml
import lxml.etree
import requests

url='http://openAPI.seoul.go.kr:8088/73725974496a736c34395757705847/xml/SearchSTNBySubwayLineInfo/1/10///2'
data=requests.get(url).text  #requests.get(url)은 response임. 따라서 requests.get(url).text는 response.text와 같음
tree=lxml.etree.fromstring(data.encode('utf-8')) #string은 문자열/ parse는 파일

In [31]:
for node in tree.xpath('//STATION_NM'): #//역할 : STATION이 어디 있든 지 간에 호출
    print(node.text)

명학
군포
부개
소사
구일
인천
동인천
제물포
주안
동암


In [32]:
for node in tree.xpath('//list_total_count'):
    list_total_count=int(node.text)
    print ("total_count=",list_total_count, node.text)

total_count= 799 799


In [33]:
int(list_total_count)

799

In [35]:
%%writefile src/ds_open_subway_xml.py

import os
import requests
import urllib
import mylib # NO! src.mylib
import lxml
import lxml.etree
from io import StringIO

def doIt():
    keyPath=os.path.join(os.getcwd(), 'src', 'key.properties')
    key=mylib.getKey(keyPath)
    # (1) make params with resource IDs
    KEY=str(key['dataseoul'])
    TYPE='xml'
    #OLD: SERVICE='SearchSTNBySubwayLineService'
    SERVICE='SearchSTNBySubwayLineInfo'
    LINE_NUM=str(2)
    START_INDEX=str(1)
    END_INDEX=str(10)
    startIndex=1
    endIndex=10
    list_total_count=0    # set later
    while True:
        START_INDEX=str(startIndex)
        END_INDEX=str(endIndex)
        #params=os.path.join(KEY,TYPE,SERVICE,START_INDEX,END_INDEX,LINE_NUM)
        params="/".join([KEY,TYPE,SERVICE,START_INDEX,END_INDEX,'','',LINE_NUM])
        # (2) make a full url
        _url='http://openAPI.seoul.go.kr:8088' #NOTE slash: do not use 'http://openAPI.seoul.go.kr:8088/'
        #url=urllib.parse.urljoin(_url,params)
        url="/".join([_url,params])
        #print(url)
        # (3) get data
            data=requests.get(url).text
            #tree=lxml.etree.parse(StringIO(data.encode('utf-8')))
            tree=lxml.etree.fromstring(data.encode('utf-8'))
            if(startIndex==1):
                for node in tree.xpath('//list_total_count'):
                    list_total_count=int(node.text)
                    print ("- Total_count=",list_total_count)
            for node in tree.xpath('//STATION_NM'):
            startIndex+=10
            endIndex+=10
            if(endIndex > list_total_count):
                print("----- Ending endIndex=", endIndex)
                break

if __name__ == "__main__":
    doIt()

Overwriting src/ds_open_subway_xml.py


In [36]:
!python src/ds_open_subway_xml.py #202609

- Total_count= 799
명학, 군포, 부개, 소사, 구일, 인천, 동인천, 제물포, 주안, 동암, 백운, 부평, 오류동, 개봉, 천안, 두정, 직산, 성환, 진위, 수원, 화서, 창동, 남영, 용산, 노량진, 대방, 영등포, 신도림, 봉명, 쌍용, 아산, 탕정, 배방, 외대앞, 신이문, 온양온천, 신창, 동묘앞, 청량리, 제기동, 신설동, 동대문, 종로5가, 종로3가, 종각, 시청, 서울역, 지행, 송내, 부천, 역곡, 망월사, 석계, 광운대, 도봉산, 월계, 도봉, 방학, 광명, 서동탄, 가산디지털단지, 금천구청, 당정, 녹천, 석수, 관악, 평택, 평택지제, 서정리, 구로, 동두천중앙, 동두천, 소요산, 청산, 전곡, 성균관대, 의왕, 송탄, 오산, 오산대, 세마, 병점, 세류, 독산, 안양, 연천, 회룡, 의정부, 가능, 녹양, 양주, 덕계, 덕정, 보산, 회기, 금정, 신길, 도화, 중동, 온수, 도원, 간석, 강변, 상왕십리, 역삼, 서초, 성수, 용두, 신정네거리, 양천구청, 도림천, 신설동, 신답, 용답, 동대문역사문화공원, 을지로입구, 교대, 시청, 충정로, 아현, 이대, 신촌, 홍대입구, 왕십리, 한양대, 뚝섬, 신림, 잠실나루, 신당, 합정, 당산, 을지로4가, 잠실, 잠실새내, 강남, 까치산, 영등포구청, 문래, 신도림, 대림, 낙성대, 구로디지털단지, 사당, 봉천, 서울대입구, 건대입구, 방배, 종합운동장, 삼성, 선릉, 을지로3가, 신대방, 구의, 정발산, 화정, 대치, 대화, 주엽, 가락시장, 신사, 압구정, 삼송, 약수, 금호, 고속터미널, 을지로3가, 도곡, 매봉, 양재, 종로3가, 수서, 일원, 원당, 대청, 옥수, 학여울, 원흥, 남부터미널, 교대, 안국, 독립문, 무악재, 홍제, 오금, 연신내, 구파발, 지축, 동대입구, 충무로, 잠원, 마두, 녹번, 불광, 경복궁, 백석, 대곡, 경찰병원, 회현, 오이도, 정왕, 사당, 남태령, 금정, 범계, 평촌, 인덕원, 총신대입구, 동작, 이촌, 신용산, 진접, 오남, 별내별

## 문제3) 행정동별 서울생활인구

#### 일괄실행

In [38]:
%%writefile src/ds_open_foreigners_xml.py
import os
import urllib
import requests
import lxml
import lxml.etree
import mylib #from src import mylib

def doIt():
    keyPath=os.path.join(os.getcwd(), 'src', 'key.properties')
    key=mylib.getKey(keyPath)
    KEY=str(key['dataseoul'])
    TYPE='xml'
    SERVICE='SPOP_FORN_TEMP_RESD_DONG'
    START_INDEX=str(1)
    END_INDEX=str(10)
    STDR_DE_ID=str(20200617) #일자는 문자열로 읽어야됨
    #params="/".join([KEY,TYPE,SERVICE,START_INDEX,END_INDEX,STDR_DE_ID])

    #_url='http://openAPI.seoul.go.kr:8088/'
    _url='http://openAPI.seoul.go.kr:8088' # NOTE: the slash at the end removed
    #url=urllib.parse.urljoin(_url,params)
    url="/".join([_url,KEY,TYPE,SERVICE,START_INDEX,END_INDEX,STDR_DE_ID])
    data=requests.get(url).text
    #tree=lxml.etree.parse(StringIO.StringIO(data.encode('utf-8')))
    tree=lxml.etree.fromstring(data.encode('utf-8'))
    nodes=tree.xpath('//TOT_LVPOP_CO')
    for node in nodes:
        print (node.text)

if __name__ == "__main__":
    doIt()

Writing src/ds_open_foreigners_xml.py


In [40]:
!python src/ds_open_foreigners_xml.py #202609

248.866
71.2628
150.1777
136.7137
174.824
0.4833
0.2163
35.2093
756.5433
87.6117


## 문제5) 서울시 지하철 호선별 역별 시간대별 승하차 인원 정보

In [42]:
#인증키
import os
from src import mylib

keyPath=os.path.join(os.getcwd(),'src','key.properties')
key=mylib.getKey(keyPath)

In [49]:
#요청인자
_url='http://openAPI.seoul.go.kr:8088'
_key=str(key['dataseoul'])
_type='xml'
_service='CardSubwayTime'
_start_index=1
_end_index=5
_use_mon='202106'

In [50]:
import requests

_maxIter=1
_iter=0
while _iter<_maxIter:
     _api="/".join([_url,_key,_type,_service,str(_start_index),str(_end_index),_use_mon])
     
     response=requests.get(_api).text
     print(response)
     _start_index+=5
     _end_index+=5
     _iter+=1


<?xml version="1.0" encoding="UTF-8"?>
<CardSubwayTime>
<list_total_count>608</list_total_count>
<RESULT>
<CODE>INFO-000</CODE>
<MESSAGE>정상 처리되었습니다</MESSAGE>
</RESULT>
<row>
<USE_MM>202106</USE_MM>
<SBWY_ROUT_LN_NM>1호선</SBWY_ROUT_LN_NM>
<STTN>서울역</STTN>
<HR_4_GET_ON_NOPE>654</HR_4_GET_ON_NOPE>
<HR_4_GET_OFF_NOPE>17</HR_4_GET_OFF_NOPE>
<HR_5_GET_ON_NOPE>9008</HR_5_GET_ON_NOPE>
<HR_5_GET_OFF_NOPE>6400</HR_5_GET_OFF_NOPE>
<HR_6_GET_ON_NOPE>12474</HR_6_GET_ON_NOPE>
<HR_6_GET_OFF_NOPE>37203</HR_6_GET_OFF_NOPE>
<HR_7_GET_ON_NOPE>37253</HR_7_GET_ON_NOPE>
<HR_7_GET_OFF_NOPE>91875</HR_7_GET_OFF_NOPE>
<HR_8_GET_ON_NOPE>59876</HR_8_GET_ON_NOPE>
<HR_8_GET_OFF_NOPE>187805</HR_8_GET_OFF_NOPE>
<HR_9_GET_ON_NOPE>44619</HR_9_GET_ON_NOPE>
<HR_9_GET_OFF_NOPE>118679</HR_9_GET_OFF_NOPE>
<HR_10_GET_ON_NOPE>42611</HR_10_GET_ON_NOPE>
<HR_10_GET_OFF_NOPE>57710</HR_10_GET_OFF_NOPE>
<HR_11_GET_ON_NOPE>49533</HR_11_GET_ON_NOPE>
<HR_11_GET_OFF_NOPE>50003</HR_11_GET_OFF_NOPE>
<HR_12_GET_ON_NOPE>59357</HR_12_GET_ON_

##### json 반환

# 문제7) DB저장

In [ ]:
서버를 띄울 땐, Jupyter에서 하는 게 아닌 서버(cmd)를 띄워야 함

In [ ]:
%%writefile src/Foreigner

import os
import mylib
import urllib
import requests

#1. Key값 저장
keyPath=os.path.join(os.getcwd(),'src','src.properties')
key=mylib.getKey(keyPath)
KEY=str(key['dataseoul'])

#2. API값
TYPE='json'
SERVICE='Spop250mFornTempDong'
START_INDEX=1
END_INDEX=5

params='/'.join([KEY,TYPE,SERVICE,START_INDEX,END_INDEX])

#3. URL
_url='https://data.seoul.go.kr'
url='/'.join([_url,params])

#4. DATA 불러오기
r=requests.get([_url,params])
Foreigner=r.json()

for e in [Spop250mFornTempDong][rows]:
    print(u

In [1]:
!mongosh --version

2.13.0
